## Training + Hyperparameter tuning + Evaluation - NB

### Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold, RandomizedSearchCV


from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    log_loss
)

from nboost import nbClassifier
from scipy.stats import uniform, randint, loguniform

random_state = 42

### Load datasets

In [ ]:
both_df = pd.read_csv("both_intervals_features.csv")
p600_df = pd.read_csv("p600_features.csv")
p600_df = pd.read_csv("p600_features.csv")

### Splitting

In [ ]:
def split(df, random_state):
    df = df.copy() # copy to not break anything

    df["label"] = (df["Cloze"] > 0).astype(int) # cloze = 0 -> class 0 else class = 1

    cloze = df["Cloze"]  # keep it for the cloze performance analysis

    X = df.drop(columns=["Cloze", "label", "trial_global", "Trial", "site", "subject_global"]) # only keep features for training
    y = df["label"] # target variable 
    groups = df["subject_global"] # for subject-based splitting and grid search

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state) # 80-20 split with subjects as group to avoid data leakage
    train_idx, test_idx = next(gss.split(X, y, groups))

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train = groups.iloc[train_idx]
    groups_test = groups.iloc[test_idx]

    cloze_test = cloze.iloc[test_idx]  # also split original cloze to keep track of shuffling and splitting of each seed

    return X_train, X_test, y_train, y_test, groups_train, groups_test, cloze_test

Naive Bayes does not have hyperparameters, but still adding this grid for a consistent pipeline with the other algorithms.

In [11]:
parameter_grid = {
    "model__var_smoothing": [1e-9]  # default
}

### Select most frequent parameters

In [ ]:
# identify the most frequently occurring CONFIGURATION
def select_best_params(best_params_list):
    params_tuple = [tuple(sorted(p.items())) for p in best_params_list]
    most_common = Counter(params_tuple).most_common(1)[0][0]
    return dict(most_common)

### Nested CV

In [ ]:

# Nested cross-validation and grid search on the training data with subject-based grouping
def nested_cv_nb(X_train, y_train, groups_train, parameter_grid, random_state):
    outer_cv = GroupKFold(n_splits=5) # outer split into folds for evaluation of chosen hyperparameters within the inner folds

    outer_scores = [] # keep track of the scores and associated hyperparam.
    best_params_list = []

    pipeline = Pipeline([
        ("scaler", StandardScaler()), # kept for consistency with the LR pipeline, though NB is scale-invariant
        ("model", GaussianNB())
    ])

    # grid search loop for the inner folds
    for outer_train_idx, outer_val_idx in outer_cv.split(X_train, y_train, groups_train):

        # extract the training and validation folds for each iteration
        X_outer_train = X_train.iloc[outer_train_idx]
        X_outer_val   = X_train.iloc[outer_val_idx]

        y_outer_train = y_train.iloc[outer_train_idx]
        y_outer_val   = y_train.iloc[outer_val_idx]

        # also split the groups
        groups_outer_train = groups_train.iloc[outer_train_idx]

        inner_cv = GroupKFold(n_splits=5)

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=parameter_grid,
            cv=inner_cv,
            scoring="balanced_accuracy", # select best hyperparameters based on balanced accuracy
            n_jobs=-1
        )

        # run grid search, keeping track of groups
        grid_search.fit(X_outer_train, y_outer_train, groups=groups_outer_train)

        # retrieve the best model found by inner CV
        best_model = grid_search.best_estimator_
        best_params_list.append(grid_search.best_params_) # there is only one dummy parameter

        # evaluate on outer fold
        y_pred = best_model.predict(X_outer_val)
        score = balanced_accuracy_score(y_outer_val, y_pred)
        outer_scores.append(score)

    return (outer_scores, select_best_params(best_params_list))

### Final train and test function

In [ ]:
def evaluate_nb(X_train, X_test, y_train, y_test, best_params, random_state):

    final_nb = Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianNB(
            var_smoothing=best_params["model__var_smoothing"]
        ))
    ])

    # fit on training data
    final_nb.fit(X_train, y_train)

    y_pred = final_nb.predict(X_test) # labels
    y_prob = final_nb.predict_proba(X_test)[:, 1] # probabilities 
    test_score_nb = balanced_accuracy_score(y_test, y_pred)

    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": test_score_nb,
        "f1": f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "test_score_nb": test_score_nb,
        "classification_report": classification_report(y_test, y_pred),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "y_pred": y_pred,
        "y_prob": y_prob
    }

    return final_nb, results

### Both intervals (P6 + N4) - the entire pipeline

In [ ]:
import json
import joblib
from collections import Counter

results_list = [] # keep track of all the results and params
all_params = []

# Run the entire pipeline across 10 different seeds to investigate how stable the results are
for seed in range(10):
    print(f"\nRunning seed {seed}...")

    # split the data (take seed into account)
    X_train_both, X_test_both, y_train_both, y_test_both, groups_train_both, groups_test_both, cloze_test_both = split(both_df, seed)

    # nested CV for the selection of the hyperparameters
    outer_scores_both, best_params_both = nested_cv_nb(
        X_train_both, y_train_both, groups_train_both, parameter_grid, seed
    )

    # evaluate the final model with the best hyperparameters on the current seed
    final_nb_both, results_both = evaluate_nb(
        X_train_both, X_test_both, y_train_both, y_test_both, best_params_both, seed
    )

    # store the results per seed
    results_list.append({
        "seed": seed,
        "accuracy": results_both["accuracy"],
        "balanced_accuracy": results_both["balanced_accuracy"],
        "f1": results_both["f1"],
        "precision": results_both["precision"],
        "recall": results_both["recall"],
        "roc_auc": results_both["roc_auc"],
        "best_params": best_params_both
    })

    all_params.append(best_params_both) # also save the parameters associated with each seed

    temp_df = pd.DataFrame(results_list)
    temp_df.to_csv("nb_results_running_both.csv", index=False)

    # Save predictions for this seed for performance as a function of cloze analysis
    pred_df = pd.DataFrame({
        "y_true": y_test_both.values,
        "y_pred": results_both["y_pred"],
        "y_prob": results_both["y_prob"],
        "Cloze": cloze_test_both.values,
        "Seed": seed
    })
    pred_df.to_csv(f"nb_predictions_seed_{seed}.csv", index=False)

    # Save model for this seed
    joblib.dump(final_nb_both, f"nb_model_seed_{seed}.joblib")

    print(f"Seed {seed} done.")

# final save
results_df = pd.DataFrame(results_list)
results_df.to_csv("nb_results_final.csv", index=False)

# for printing summary of the results
summary = {
    "accuracy_mean": float(results_df["accuracy"].mean()),
    "accuracy_std": float(results_df["accuracy"].std()),
    "balanced_accuracy_mean": float(results_df["balanced_accuracy"].mean()),
    "balanced_accuracy_std": float(results_df["balanced_accuracy"].std()),
    "roc_auc_mean": float(results_df["roc_auc"].mean()),
    "roc_auc_std": float(results_df["roc_auc"].std())
}

with open("nb_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

# select the most frequent configuration across 10 seeds
params_tuple = [tuple(sorted(p.items())) for p in all_params]
best_params_final = dict(Counter(params_tuple).most_common(1)[0][0])

print("\nFinal Best Parameters:")
print(best_params_final)

with open("nb_best_params.json", "w") as f:
    json.dump(best_params_final, f, indent=4)

# print final results
print("\nFinal Results:")
print(results_df)

print("\nSummary:")
print(summary)


Running seed 0...
Seed 0 done.

Running seed 1...
Seed 1 done.

Running seed 2...
Seed 2 done.

Running seed 3...
Seed 3 done.

Running seed 4...
Seed 4 done.

Running seed 5...
Seed 5 done.

Running seed 6...
Seed 6 done.

Running seed 7...
Seed 7 done.

Running seed 8...
Seed 8 done.

Running seed 9...
Seed 9 done.

Final Best Parameters:
{'model__var_smoothing': 1e-09}

Final Results:
   seed  accuracy  balanced_accuracy        f1  precision    recall   roc_auc  \
0     0  0.550464           0.550408  0.538117   0.527473  0.549199  0.577040   
1     1  0.581452           0.576571  0.526900   0.561466  0.496343  0.619397   
2     2  0.589354           0.588579  0.610309   0.590229  0.631804  0.611247   
3     3  0.565473           0.565933  0.562668   0.547071  0.579181  0.590485   
4     4  0.582781           0.582634  0.590667   0.591716  0.589623  0.615801   
5     5  0.567052           0.567166  0.563323   0.573196  0.553785  0.597956   
6     6  0.566173           0.564306  0.4

### P600 interval - the entire pipeline (comments only on the combined data above, but the code is exactly the same, just a differrent dataset)

In [ ]:
import json
import joblib
from collections import Counter

results_list_nb_p6 = []
all_params_nb_p6 = []

for seed in range(10):
    print(f"\nRunning seed {seed}...")

    X_train_p6_nb, X_test_p6_nb, y_train_p6_nb, y_test_p6_nb, groups_train_p6_nb, groups_test_p6_nb, cloze_test_nb_p6 = split(p600_df, seed)

    outer_scores_p6_nb, best_params_p6_nb = nested_cv_nb(
        X_train_p6_nb, y_train_p6_nb, groups_train_p6_nb, parameter_grid, seed
    )

    final_nb_p6, results_p6_nb = evaluate_nb(
        X_train_p6_nb, X_test_p6_nb, y_train_p6_nb, y_test_p6_nb, best_params_p6_nb, seed
    )

    results_list_nb_p6.append({
        "seed": seed,
        "accuracy": results_p6_nb["accuracy"],
        "balanced_accuracy": results_p6_nb["balanced_accuracy"],
        "f1": results_p6_nb["f1"],
        "precision": results_p6_nb["precision"],
        "recall": results_p6_nb["recall"],
        "roc_auc": results_p6_nb["roc_auc"],
        "best_params": best_params_p6_nb
    })

    all_params_nb_p6.append(best_params_p6_nb)

    temp_df = pd.DataFrame(results_list_nb_p6)
    temp_df.to_csv("nb_results_p6_running.csv", index=False)

    pred_df_nb_p6 = pd.DataFrame({
        "y_true": y_test_p6_nb,
        "y_pred": results_p6_nb["y_pred"],
        "y_prob": results_p6_nb["y_prob"]
    })
    pred_df_nb_p6.to_csv(f"nb_predictions_seed_{seed}_p6.csv", index=False)

    joblib.dump(final_nb_p6, f"nb_model_seed_{seed}_p6.joblib")

    print(f"Seed {seed} done.")

results_df_nb_p6 = pd.DataFrame(results_list_nb_p6)
results_df_nb_p6.to_csv("nb_results_final_p6.csv", index=False)

summary_nb_p6 = {
    "accuracy_mean": float(results_df_nb_p6["accuracy"].mean()),
    "accuracy_std": float(results_df_nb_p6["accuracy"].std()),
    "balanced_accuracy_mean": float(results_df_nb_p6["balanced_accuracy"].mean()),
    "balanced_accuracy_std": float(results_df_nb_p6["balanced_accuracy"].std()),
    "roc_auc_mean": float(results_df_nb_p6["roc_auc"].mean()),
    "roc_auc_std": float(results_df_nb_p6["roc_auc"].std())
}

with open("nb_p6_summary.json", "w") as f:
    json.dump(summary_nb_p6, f, indent=4)

params_tuple_nb_p6 = [tuple(sorted(p.items())) for p in all_params_nb_p6]
best_params_final_nb_p6 = dict(Counter(params_tuple_nb_p6).most_common(1)[0][0])

print("\nFinal Best Parameters:")
print(best_params_final_nb_p6)

with open("nb_p6_best_params.json", "w") as f:
    json.dump(best_params_final_nb_p6, f, indent=4)

print("\nFinal Results:")
print(results_df_nb_p6)

print("\nSummary:")
print(summary_nb_p6)


Running seed 0...
Seed 0 done.

Running seed 1...
Seed 1 done.

Running seed 2...
Seed 2 done.

Running seed 3...
Seed 3 done.

Running seed 4...
Seed 4 done.

Running seed 5...
Seed 5 done.

Running seed 6...
Seed 6 done.

Running seed 7...
Seed 7 done.

Running seed 8...
Seed 8 done.

Running seed 9...
Seed 9 done.

Final Best Parameters:
{'model__var_smoothing': 1e-09}

Final Results:
   seed  accuracy  balanced_accuracy        f1  precision    recall   roc_auc  \
0     0  0.521004           0.523876  0.538381   0.498054  0.585812  0.534301   
1     1  0.557409           0.556963  0.538383   0.527583  0.549634  0.575859   
2     2  0.554590           0.552966  0.595262   0.553719  0.643543  0.570301   
3     3  0.545697           0.548383  0.570707   0.524605  0.625692  0.553949   
4     4  0.555087           0.553202  0.595954   0.555556  0.642689  0.572914   
5     5  0.533903           0.533577  0.552987   0.535448  0.571713  0.551437   
6     6  0.542156           0.541547  0.5

### N400 interval - the entire pipeline (comments only on the combined data above, but the code is exactly the same, just a differrent dataset)

In [ ]:
import json
import joblib
from collections import Counter

results_list_nb_n4 = []
all_params_nb_n4 = []

for seed in range(10):
    print(f"\nRunning seed {seed}...")

    X_train_n4_nb, X_test_n4_nb, y_train_n4_nb, y_test_n4_nb, groups_train_n4_nb, groups_test_n4_nb, cloze_test_n4 = split(n400_df, seed)

    outer_scores_n4_nb, best_params_n4_nb = nested_cv_nb(
        X_train_n4_nb, y_train_n4_nb, groups_train_n4_nb, parameter_grid, seed
    )

    final_nb_n4, results_n4_nb = evaluate_nb(
        X_train_n4_nb, X_test_n4_nb, y_train_n4_nb, y_test_n4_nb, best_params_n4_nb, seed
    )

    results_list_nb_n4.append({
        "seed": seed,
        "accuracy": results_n4_nb["accuracy"],
        "balanced_accuracy": results_n4_nb["balanced_accuracy"],
        "f1": results_n4_nb["f1"],
        "precision": results_n4_nb["precision"],
        "recall": results_n4_nb["recall"],
        "roc_auc": results_n4_nb["roc_auc"],
        "best_params": best_params_n4_nb
    })

    all_params_nb_n4.append(best_params_n4_nb)

    temp_df = pd.DataFrame(results_list_nb_n4)
    temp_df.to_csv("nb_results_n4_running.csv", index=False)

    pred_df_nb_n4 = pd.DataFrame({
        "y_true": y_test_n4_nb,
        "y_pred": results_n4_nb["y_pred"],
        "y_prob": results_n4_nb["y_prob"]
    })
    pred_df_nb_n4.to_csv(f"nb_predictions_seed_{seed}_n4.csv", index=False)

    joblib.dump(final_nb_n4, f"nb_model_seed_{seed}_n4.joblib")

    print(f"Seed {seed} done.")

results_df_nb_n4 = pd.DataFrame(results_list_nb_n4)
results_df_nb_n4.to_csv("nb_results_final_n4.csv", index=False)

summary_nb_n4 = {
    "accuracy_mean": float(results_df_nb_n4["accuracy"].mean()),
    "accuracy_std": float(results_df_nb_n4["accuracy"].std()),
    "balanced_accuracy_mean": float(results_df_nb_n4["balanced_accuracy"].mean()),
    "balanced_accuracy_std": float(results_df_nb_n4["balanced_accuracy"].std()),
    "roc_auc_mean": float(results_df_nb_n4["roc_auc"].mean()),
    "roc_auc_std": float(results_df_nb_n4["roc_auc"].std())
}

with open("nb_n4_summary.json", "w") as f:
    json.dump(summary_nb_n4, f, indent=4)

params_tuple_nb_n4 = [tuple(sorted(p.items())) for p in all_params_nb_n4]
best_params_final_nb_n4 = dict(Counter(params_tuple_nb_n4).most_common(1)[0][0])

print("\nFinal Best Parameters:")
print(best_params_final_nb_n4)

with open("nb_n4_best_params.json", "w") as f:
    json.dump(best_params_final_nb_n4, f, indent=4)

print("\nFinal Results:")
print(results_df_nb_n4)

print("\nSummary:")
print(summary_nb_n4)


Running seed 0...
Seed 0 done.

Running seed 1...
Seed 1 done.

Running seed 2...
Seed 2 done.

Running seed 3...
Seed 3 done.

Running seed 4...
Seed 4 done.

Running seed 5...
Seed 5 done.

Running seed 6...
Seed 6 done.

Running seed 7...
Seed 7 done.

Running seed 8...
Seed 8 done.

Running seed 9...
Seed 9 done.

Final Best Parameters:
{'model__var_smoothing': 1e-09}

Final Results:
   seed  accuracy  balanced_accuracy        f1  precision    recall   roc_auc  \
0     0  0.539553           0.536228  0.490338   0.519182  0.464531  0.561953   
1     1  0.553974           0.549950  0.504632   0.527335  0.483804  0.575728   
2     2  0.536665           0.537344  0.523197   0.549296  0.499466  0.561204   
3     3  0.531267           0.529380  0.494524   0.515625  0.475083  0.549408   
4     4  0.546659           0.548475  0.510085   0.568940  0.462264  0.574672   
5     5  0.562029           0.562375  0.545833   0.572052  0.521912  0.579601   
6     6  0.544711           0.543080  0.4